In [1]:
!pip install -q -U transformers && pip install -q accelerate sentencepiece torchvision pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 97.9 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 89.4 MB/s eta 0:00:00:00:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompa

In [3]:
from huggingface_hub import login, whoami

login()
print(whoami())

{'type': 'user', 'id': '69876798b92dcf60f8975a59', 'name': 'BaeBaeBoo1010', 'fullname': 'Khánh Lê', 'email': 'khanhle14062006@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1782864000, 'isPro': False, 'avatarUrl': '/avatars/44e75d6df94b8c89c22e5274755a2c03.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'test-qwen', 'role': 'read', 'createdAt': '2026-06-15T04:08:40.420Z'}}}


In [4]:
import json

path = "/kaggle/input/datasets/khanhle1406/public-test-1780368312-json/public-test_1780368312.json"

with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Số câu hỏi:", len(data))

Số câu hỏi: 463


# Load Model

In [5]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import json
import re
import torch
import gc
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoProcessor, AutoModelForImageTextToText, LogitsProcessor, LogitsProcessorList

# =====================================================
# 1. BỘ KHÓA TOKEN (OPTIMIZED)
# =====================================================
class ChoiceLogitsProcessor(LogitsProcessor):
    def __init__(self, tokenizer, choices="ABCDEFGHIJKLMNOPQRSTUVWXYZ"):
        self.allowed_token_ids = set()
        vocab = tokenizer.get_vocab()
        for token_str, token_id in vocab.items():
            clean_token = token_str.replace(' ', '').replace('Ġ', '').replace('\n', '').replace('<0x0A>', '').strip()
            if len(clean_token) == 1 and clean_token.upper() in choices:
                self.allowed_token_ids.add(token_id)
        
        for char in choices:
            for prefix in ["", " ", "\n"]:
                for c in [char, char.lower()]:
                    tids = tokenizer.encode(prefix + c, add_special_tokens=False)
                    if tids:
                        self.allowed_token_ids.add(tids[-1])
                        
        self.allowed_token_ids = list(self.allowed_token_ids)
        self.mask = None
        
    def __call__(self, input_ids, scores):
        # Optimizing mask creation: avoid allocating new memory and running multiplication on every step
        if self.mask is None or self.mask.device != scores.device:
            self.mask = torch.full((scores.size(-1),), float('-inf'), device=scores.device)
            self.mask[self.allowed_token_ids] = 0.0
        return scores + self.mask

# =====================================================
# 2. DỌN DẸP VRAM & KHỞI TẠO MODEL (OPTIMIZED)
# =====================================================
# Giải phóng bộ nhớ của các lần chạy trước để tránh lỗi CUDA Out of Memory (OOM) trong Jupyter
for var_name in ['models', 'model', 'processors', 'processor', 'inputs', 'outputs', 'outputs_step1', 'inputs_step2', 'outputs_step2']:
    if var_name in globals():
        print(f"Xóa biến hệ thống cũ: {var_name}")
        del globals()[var_name]

gc.collect()
torch.cuda.empty_cache()

MODEL_NAME = "Qwen/Qwen3.5-4B"

num_gpus = torch.cuda.device_count()
devices = [f"cuda:{i}" for i in range(num_gpus)] if num_gpus > 0 else ["cpu"]
models = []
processors = []

print(f"Phát hiện {len(devices)} thiết bị. Đang tải model và processor...")
for device in devices:
    # Tải processor riêng biệt cho mỗi GPU để tránh tranh chấp bộ nhớ (lỗi Rust "Already borrowed" của Fast Tokenizer)
    proc = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
    proc.tokenizer.padding_side = "left"
    if proc.tokenizer.pad_token is None:
        proc.tokenizer.pad_token = proc.tokenizer.eos_token
    processors.append(proc)

    # Load model trên từng GPU độc lập
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True,
        dtype=torch.float16,
        device_map={"": device},
        low_cpu_mem_usage=True,
        attn_implementation="sdpa"
    )
    model.eval()
    models.append(model)
print("✅ Tải model và processor hoàn tất!")

# =====================================================

Phát hiện 2 thiết bị. Đang tải model và processor...


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

✅ Tải model và processor hoàn tất!


# Load questions and answers

In [9]:
# 3. LOAD DATA & PHÂN LOẠI
# =====================================================
GROUND_TRUTH = {
    "test_0001": "A", "test_0002": "B", "test_0003": "B",
    "test_0004": "C", "test_0005": "C", "test_0006": "A",
    "test_0007": "B", "test_0008": "B", "test_0009": "C",
    "test_0010": "D", "test_0011": "C", "test_0012": "A",
    "test_0013": "A", "test_0014": "B", "test_0015": "C",
    "test_0016": "A", "test_0017": "B", "test_0018": "B",
    "test_0019": "D", "test_0020": "A", "test_0021": "E",
    "test_0022": "D", "test_0023": "C", "test_0024": "C",
    "test_0025": "B", "test_0026": "B", "test_0027": "A",
    "test_0028": "A", "test_0029": "D", "test_0030": "B",
    "test_0031": "A", "test_0032": "A", "test_0033": "C",
    "test_0034": "A", "test_0035": "D", "test_0036": "A",
    "test_0037": "F", "test_0038": "B", "test_0039": "A",
    "test_0040": "A", "test_0041": "A", "test_0042": "A",
    "test_0043": "E", "test_0044": "A", "test_0045": "B",
    "test_0046": "B", "test_0047": "A", "test_0048": "A",
    "test_0049": "B", "test_0050": "A", "test_0051": "B",
    "test_0052": "A", "test_0053": "B", "test_0054": "D",
    "test_0055": "B", "test_0056": "A", "test_0057": "C",
    "test_0058": "B", "test_0059": "B", "test_0060": "B",
    "test_0061": "B", "test_0062": "D", "test_0063": "B",
    "test_0064": "B", "test_0065": "D", "test_0066": "B",
    "test_0067": "A", "test_0068": "B", "test_0069": "A",
    "test_0070": "C", "test_0071": "C", "test_0072": "C",
    "test_0073": "C", "test_0074": "B", "test_0075": "C",
    "test_0076": "D", "test_0077": "A", "test_0078": "A",
    "test_0079": "B", "test_0080": "B", "test_0081": "C",
    "test_0082": "D", "test_0083": "B", "test_0084": "A",
    "test_0085": "C", "test_0086": "C", "test_0087": "B",
    "test_0088": "B", "test_0089": "B", "test_0090": "A",
    "test_0091": "B", "test_0092": "A", "test_0093": "A",
    "test_0094": "C", "test_0095": "A", "test_0096": "A",
    "test_0097": "A", "test_0098": "A", "test_0099": "B",
    "test_0100": "D", "test_0101": "C", "test_0102": "C",
    "test_0103": "B", "test_0104": "B", "test_0105": "C",
    "test_0106": "C", "test_0107": "B", "test_0108": "C",
    "test_0109": "A", "test_0110": "A", "test_0111": "B",
    "test_0112": "G", "test_0113": "C", "test_0114": "A",
    "test_0115": "D", "test_0116": "A", "test_0117": "A",
    "test_0118": "E", "test_0119": "A", "test_0120": "C",
    "test_0121": "C", "test_0122": "A", "test_0123": "B",
    "test_0124": "B", "test_0125": "D", "test_0126": "B",
    "test_0127": "A", "test_0128": "A", "test_0129": "A",
    "test_0130": "A", "test_0131": "C", "test_0132": "B",
    "test_0133": "A", "test_0134": "B", "test_0135": "B",
    "test_0136": "B", "test_0137": "C", "test_0138": "A",
    "test_0139": "D", "test_0140": "C", "test_0141": "A",
    "test_0142": "C", "test_0143": "B", "test_0144": "A",
    "test_0145": "A", "test_0146": "C", "test_0147": "E",
    "test_0148": "B", "test_0149": "C", "test_0150": "C",
    "test_0151": "F", "test_0152": "A", "test_0153": "D",
    "test_0154": "C", "test_0155": "A", "test_0156": "B",
    "test_0157": "B", "test_0158": "A", "test_0159": "D",
    "test_0160": "D", "test_0161": "A", "test_0162": "C",
    "test_0163": "D", "test_0164": "A", "test_0165": "B",
    "test_0166": "B", "test_0167": "C", "test_0168": "D",
    "test_0169": "B", "test_0170": "A", "test_0171": "B",
    "test_0172": "B", "test_0173": "D", "test_0174": "B",
    "test_0175": "B", "test_0176": "A", "test_0177": "A",
    "test_0178": "A", "test_0179": "B", "test_0180": "A",
    "test_0181": "A", "test_0182": "C", "test_0183": "B",
    "test_0184": "D", "test_0185": "C", "test_0186": "C",
    "test_0187": "D", "test_0188": "B", "test_0189": "B",
    "test_0190": "D", "test_0191": "B", "test_0192": "B",
    "test_0193": "B", "test_0194": "C", "test_0195": "B",
    "test_0196": "A", "test_0197": "B", "test_0198": "A",
    "test_0199": "B", "test_0200": "D", "test_0201": "A",
    "test_0202": "D", "test_0203": "B", "test_0204": "A",
    "test_0205": "B", "test_0206": "D", "test_0207": "B",
    "test_0208": "B", "test_0209": "D", "test_0210": "B",
    "test_0211": "A", "test_0212": "A", "test_0213": "D",
    "test_0214": "B", "test_0215": "A", "test_0216": "B",
    "test_0217": "B", "test_0218": "A", "test_0219": "B",
    "test_0220": "J", "test_0221": "A", "test_0222": "C",
    "test_0223": "C", "test_0224": "B", "test_0225": "C",
    "test_0226": "A", "test_0227": "A", "test_0228": "J",
    "test_0229": "D", "test_0230": "B", "test_0231": "B",
    "test_0232": "A", "test_0233": "A", "test_0234": "A",
    "test_0235": "B", "test_0236": "A", "test_0237": "A",
    "test_0238": "B", "test_0239": "B", "test_0240": "B",
    "test_0241": "D", "test_0242": "D", "test_0243": "B",
    "test_0244": "A", "test_0245": "E", "test_0246": "E",
    "test_0247": "D", "test_0248": "B", "test_0249": "B",
    "test_0250": "D", "test_0251": "H", "test_0252": "C",
    "test_0253": "D", "test_0254": "B", "test_0255": "C",
    "test_0256": "C", "test_0257": "C", "test_0258": "B",
    "test_0259": "A", "test_0260": "B", "test_0261": "B",
    "test_0262": "B", "test_0263": "C", "test_0264": "C",
    "test_0265": "A", "test_0266": "A", "test_0267": "B",
    "test_0268": "B", "test_0269": "I", "test_0270": "D",
    "test_0271": "D", "test_0272": "A", "test_0273": "D",
    "test_0274": "A", "test_0275": "A", "test_0276": "B",
    "test_0277": "A", "test_0278": "E", "test_0279": "B",
    "test_0280": "B", "test_0281": "A", "test_0282": "A",
    "test_0283": "A", "test_0284": "C", "test_0285": "B",
    "test_0286": "C", "test_0287": "D", "test_0288": "B",
    "test_0289": "C", "test_0290": "A", "test_0291": "B",
    "test_0292": "B", "test_0293": "D", "test_0294": "C",
    "test_0295": "A", "test_0296": "B", "test_0297": "D",
    "test_0298": "B", "test_0299": "B", "test_0300": "A",
    "test_0301": "A", "test_0302": "A", "test_0303": "A",
    "test_0304": "B", "test_0305": "C", "test_0306": "A",
    "test_0307": "H", "test_0308": "B", "test_0309": "B",
    "test_0310": "E", "test_0311": "A", "test_0312": "A",
    "test_0313": "A", "test_0314": "A", "test_0315": "A",
    "test_0316": "D", "test_0317": "C", "test_0318": "C",
    "test_0319": "A", "test_0320": "D", "test_0321": "C",
    "test_0322": "B", "test_0323": "A", "test_0324": "C",
    "test_0325": "A", "test_0326": "E", "test_0327": "B",
    "test_0328": "D", "test_0329": "A", "test_0330": "D",
    "test_0331": "C", "test_0332": "A", "test_0333": "B",
    "test_0334": "A", "test_0335": "A", "test_0336": "B",
    "test_0337": "B", "test_0338": "C", "test_0339": "A",
    "test_0340": "C", "test_0341": "D", "test_0342": "B",
    "test_0343": "B", "test_0344": "C", "test_0345": "A",
    "test_0346": "B", "test_0347": "B", "test_0348": "H",
    "test_0349": "C", "test_0350": "C", "test_0351": "G",
    "test_0352": "B", "test_0353": "B", "test_0354": "B",
    "test_0355": "A", "test_0356": "C", "test_0357": "C",
    "test_0358": "A", "test_0359": "B", "test_0360": "D",
    "test_0361": "A", "test_0362": "A", "test_0363": "B",
    "test_0364": "B", "test_0365": "C", "test_0366": "C",
    "test_0367": "C", "test_0368": "B", "test_0369": "A",
    "test_0370": "C", "test_0371": "B", "test_0372": "A",
    "test_0373": "B", "test_0374": "A", "test_0375": "C",
    "test_0376": "C", "test_0377": "D", "test_0378": "D",
    "test_0379": "B", "test_0380": "B", "test_0381": "E",
    "test_0382": "B", "test_0383": "B", "test_0384": "A",
    "test_0385": "B", "test_0386": "A", "test_0387": "C",
    "test_0388": "C", "test_0389": "A", "test_0390": "A",
    "test_0391": "A", "test_0392": "B", "test_0393": "B",
    "test_0394": "A", "test_0395": "A", "test_0396": "D",
    "test_0397": "E", "test_0398": "D", "test_0399": "D",
    "test_0400": "A", "test_0401": "A", "test_0402": "E",
    "test_0403": "A", "test_0404": "C", "test_0405": "C",
    "test_0406": "B", "test_0407": "B", "test_0408": "A",
    "test_0409": "B", "test_0410": "B", "test_0411": "C",
    "test_0412": "A", "test_0413": "A", "test_0414": "D",
    "test_0415": "D", "test_0416": "B", "test_0417": "D",
    "test_0418": "D", "test_0419": "A", "test_0420": "A",
    "test_0421": "A", "test_0422": "A", "test_0423": "D",
    "test_0424": "B", "test_0425": "D", "test_0426": "B",
    "test_0427": "B", "test_0428": "C", "test_0429": "B",
    "test_0430": "A", "test_0431": "C", "test_0432": "C",
    "test_0433": "B", "test_0434": "B", "test_0435": "A",
    "test_0436": "C", "test_0437": "B", "test_0438": "C",
    "test_0439": "A", "test_0440": "A", "test_0441": "B",
    "test_0442": "A", "test_0443": "C", "test_0444": "A",
    "test_0445": "B", "test_0446": "A", "test_0447": "B",
    "test_0448": "A", "test_0449": "D", "test_0450": "A",
    "test_0451": "B", "test_0452": "B", "test_0453": "A",
    "test_0454": "B", "test_0455": "B", "test_0456": "C",
    "test_0457": "D", "test_0458": "D", "test_0459": "C",
    "test_0460": "A", "test_0461": "B", "test_0462": "C",
    "test_0463": "D"
}
JSON_PATH = "/kaggle/input/datasets/khanhle1406/public-test-1780368312-json/public-test_1780368312.json"

with open(JSON_PATH, "r", encoding="utf-8") as f:
    questions = json.load(f)[200:463]

# Run Model

* Method 1

In [ ]:
LETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

def is_reasoning_question(question):
    return "là bao nhiêu" in question.lower()

reasoning_questions = [q for q in questions if is_reasoning_question(q["question"])]
knowledge_questions = [q for q in questions if not is_reasoning_question(q["question"])]

SYSTEM_KNOWLEDGE = "Chỉ trả về 1 chữ cái duy nhất (A, B, C, D, E, F, G, H, I, J)."
SYSTEM_REASONING = """Bạn là chuyên gia. Hãy suy luận giải quyết bài toán thật chi tiết. 
LƯU Ý QUAN TRỌNG: 
1. Bạn có rất nhiều khoảng trống để tính toán, KHÔNG ĐƯỢC DỪNG LẠI GIỮA CHỪNG. 
2. Nếu kết quả là phân số, biểu thức hoặc có chứa số Pi (π), BẮT BUỘC quy đổi ra số thập phân xấp xỉ để tìm đáp án gần nhất.
3. Chốt lại kết quả rõ ràng ở cuối."""

def build_messages(question_items, system_prompt):
    messages = []
    for item in question_items:
        choices_text = "\n".join(f"{LETTERS[i]}. {choice}" for i, choice in enumerate(item["choices"]))
        user_prompt = f"Câu hỏi:\n{item['question']}\n\nLựa chọn:\n{choices_text}\n\nNhiệm vụ: Chọn đúng 1 đáp án."
        messages.append([
            {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
            {"role": "user", "content": [{"type": "text", "text": user_prompt}]}
        ])
    return messages

# =====================================================
# 4. HÀM CHẠY KNOWLEDGE (TỐI ƯU HÓA)
# =====================================================
def run_knowledge_gpu(question_items, max_new_tokens, model, device, processor):
    if not question_items: return {}
    preds = {}
    BATCH_SIZE = 8  # Xử lý song song batch lớn trên GPU T4
    
    # Khởi tạo ChoiceLogitsProcessor một lần duy nhất cho mỗi luồng worker
    logits_processor = LogitsProcessorList([ChoiceLogitsProcessor(processor.tokenizer)])
    
    for i in range(0, len(question_items), BATCH_SIZE):
        mini_batch = question_items[i:i + BATCH_SIZE]
        batch_messages = build_messages(mini_batch, SYSTEM_KNOWLEDGE)
        texts = [processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True) + "Đáp án:" for msg in batch_messages]
        
        inputs = processor(text=texts, padding=True, truncation=True, return_tensors="pt").to(device)
        
        with torch.inference_mode():
            outputs = model.generate(
                **inputs, max_new_tokens=1, do_sample=False, pad_token_id=processor.tokenizer.eos_token_id, logits_processor=logits_processor
            )

        answers = processor.batch_decode(outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)
        
        for item, ans in zip(mini_batch, answers):
            raw = ans.strip()
            m = re.search(r"([A-Z])", raw.upper())
            preds[item["qid"]] = {"pred": m.group(1) if m else "A", "raw": raw}
            
        del inputs, outputs
        
    return preds

# =====================================================
# 5. HÀM CHẠY REASONING (TỐI ƯU HÓA)
# =====================================================
def run_reasoning_gpu(question_items, max_new_tokens, model, device, processor):
    if not question_items: return {}
    preds = {}
    BATCH_SIZE = 2  # Batch_size = 2 để cân đối VRAM của GPU T4 khi sinh chuỗi dài
    
    logits_processor = LogitsProcessorList([ChoiceLogitsProcessor(processor.tokenizer)])
    
    for i in range(0, len(question_items), BATCH_SIZE):
        mini_batch = question_items[i:i + BATCH_SIZE]
        batch_messages = build_messages(mini_batch, SYSTEM_REASONING)
        texts = [processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True) + "Suy luận:\n" for msg in batch_messages]
        
        # --- BƯỚC 1: SUY LUẬN ---
        inputs = processor(text=texts, padding=True, truncation=True, return_tensors="pt").to(device)
        with torch.inference_mode():
            outputs_step1 = model.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=processor.tokenizer.eos_token_id
            )
        
        reasoning_texts = processor.batch_decode(outputs_step1[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)
        
        # --- BƯỚC 2: ÉP CHỐT ---
        forced_texts = [text + "\n\nVậy đáp án đúng là:" for text in processor.batch_decode(outputs_step1, skip_special_tokens=True)]
        inputs_step2 = processor(text=forced_texts, padding=True, truncation=True, return_tensors="pt").to(device)
        
        with torch.inference_mode():
            outputs_step2 = model.generate(
                **inputs_step2, max_new_tokens=1, do_sample=False, pad_token_id=processor.tokenizer.eos_token_id, logits_processor=logits_processor
            )
        
        final_letters = processor.batch_decode(outputs_step2[:, inputs_step2.input_ids.shape[1]:], skip_special_tokens=True)
        
        for item, reasoning_part, final_letter in zip(mini_batch, reasoning_texts, final_letters):
            raw_combined = reasoning_part.strip() + "\n\n[ÉP CHỐT]: " + final_letter.strip()
            m = re.search(r"([A-Z])", final_letter.upper())
            preds[item["qid"]] = {"pred": m.group(1) if m else "A", "raw": raw_combined}
            
        del inputs, outputs_step1, inputs_step2, outputs_step2
        
    return preds

# =====================================================
# 6. TRÌNH QUẢN LÝ ĐA LUỒNG (OPTIMIZED)
# =====================================================
def run_batch(question_items, mode, max_new_tokens):
    if not question_items: return {}

    num_workers = len(models)
    # Chia danh sách lớn thành các Khối (Chunks) giao cho từng GPU
    chunk_size = (len(question_items) + num_workers - 1) // num_workers
    chunks = [question_items[i:i + chunk_size] for i in range(0, len(question_items), chunk_size)]
    
    results = {}
    
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        futures = []
        for i, chunk in enumerate(chunks):
            if chunk:
                target_model = models[i % num_workers]
                target_device = devices[i % num_workers] # Dùng chuỗi thiết bị (cuda:x)
                target_processor = processors[i % num_workers] # Dùng processor độc lập để tránh tranh chấp luồng
                func = run_knowledge_gpu if mode == "KNOWLEDGE" else run_reasoning_gpu
                futures.append(executor.submit(func, chunk, max_new_tokens, target_model, target_device, target_processor))
                
        for future in futures:
            results.update(future.result())
            
    return results

# =====================================================
# 7. THỰC THI & HIỂN THỊ
# =====================================================
results = {}

print(f"\n[1/2] Đang xử lý {len(knowledge_questions)} câu hỏi thường (Output 1 ký tự)...")
if knowledge_questions:
    results.update(run_batch(knowledge_questions, mode="KNOWLEDGE", max_new_tokens=1))

print(f"\n[2/2] Đang xử lý {len(reasoning_questions)} câu hỏi 'là bao nhiêu' (Cơ chế 2 bước, Suy luận max 1000 tokens)...")
if reasoning_questions:
    results.update(run_batch(reasoning_questions, mode="REASONING", max_new_tokens=1000))

# =====================================================
# 8. SHOW OUTPUT & EVALUATION (OPTIMIZED FOR BATCH)
# =====================================================
print("\n" + "=" * 100)
print("MODEL OUTPUT")
print("=" * 100)

for item in questions:
    qid = item["qid"]
    mode = "REASONING" if is_reasoning_question(item["question"]) else "KNOWLEDGE"
    res = results.get(qid, {"pred": "?", "raw": "NO OUTPUT"})
    
    print(f"QID   : {qid}")
    print(f"MODE  : {mode}")
    print(f"RAW   :\n{res['raw']}")
    print(f"PRED  : {res['pred']}")
    print("-" * 100)

correct = 0
total_evaluated = 0
print("\n" + "=" * 100 + "\nEVALUATION\n" + "=" * 100)
for item in questions:
    qid = item["qid"]
    if qid in GROUND_TRUTH:
        gt = GROUND_TRUTH[qid]
        pred = results.get(qid, {}).get("pred", "?")
        ok = pred == gt
        if ok:
            correct += 1
        total_evaluated += 1
        print(f"{qid:12s} | Pred={pred:2s} | GT={gt:2s} | {'✓' if ok else '✗'}")

print("=" * 100)
if total_evaluated > 0:
    print(f"Correct : {correct}/{total_evaluated}\nAccuracy: {correct / total_evaluated:.2%}")
else:
    print("No questions with ground truth were evaluated.")
print("=" * 100)


[1/2] Đang xử lý 8 câu hỏi thường (Output 1 ký tự)...

[2/2] Đang xử lý 2 câu hỏi 'là bao nhiêu' (Cơ chế 2 bước, Suy luận max 1000 tokens)...

MODEL OUTPUT
QID   : test_0031
MODE  : KNOWLEDGE
RAW   :
A
PRED  : A
----------------------------------------------------------------------------------------------------
QID   : test_0032
MODE  : KNOWLEDGE
RAW   :
A
PRED  : A
----------------------------------------------------------------------------------------------------
QID   : test_0033
MODE  : KNOWLEDGE
RAW   :
C
PRED  : C
----------------------------------------------------------------------------------------------------
QID   : test_0034
MODE  : REASONING
RAW   :
1.  **Phân tích đề bài:**
    *   Hàm tổng chi phí (TC): $TC = 100 + 5Q + 0.5Q^2$.
    *   Thị trường cạnh tranh (Perfect Competition).
    *   Yêu cầu: Tìm Chi phí biến đổi trung bình (AVC) tối thiểu tại điểm đóng cửa.
    *   Lưu ý: "Tại điểm đóng cửa" thường liên quan đến điều kiện $P = AVC$ (điểm đóng cửa ngắn hạn). Tuy nh

* Method 2

In [7]:
# =====================================================
# PHƯƠNG PHÁP 2: TỐI ƯU HÓA BẰNG LIKELIHOOD & KV CACHE
# =====================================================
import json
import re
import gc
import math
import torch
from concurrent.futures import ThreadPoolExecutor
from transformers.cache_utils import DynamicCache

# CẤU HÌNH PHƯƠNG PHÁP CHẠY LIKELIHOOD:
# True:  Phương pháp Single-Pass cực nhanh, xử lý theo BATCH (Tối ưu nhất cho tốc độ và bộ nhớ)
# False: Phương pháp Multi-Pass KV Cache truyền thống (Tuần tự từng lựa chọn, dùng DynamicCache)
USE_SINGLE_PASS = True

# BẬT/TẮT HIỆU CHUẨN KHÔNG NGỮ CẢNH (CONTENT-FREE CALIBRATION)
# Giúp triệt tiêu thiên kiến chữ cái (letter bias) và chiều dài đáp án
USE_CALIBRATION = True

# Tự động thiết lập hằng số và phân loại câu hỏi (Giúp Cell 7 độc lập với Cell 6)
LETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
SYSTEM_KNOWLEDGE = "Chỉ trả về 1 chữ cái duy nhất (A, B, C, D, E, F, G, H, I, J)."
SYSTEM_REASONING = r"""Bạn là chuyên gia giải bài tập toán, lý, hóa, kinh tế. Hãy suy luận ngắn gọn, tập trung để tìm đáp án đúng.
Quy tắc:
1. Trình bày suy luận ngắn (dưới 5 dòng), đi thẳng vào công thức, thế số và tính toán trực tiếp.
2. Chốt lại đáp án ở dòng cuối.

Ví dụ 1:
Câu hỏi: Một tụ điện 100 uF được nạp đến 120V. Hằng số thời gian xả qua điện trở 500 ohm là bao nhiêu?
Lựa chọn:
A. 0.05 giây
B. 0.1 giây
C. 0.2 giây
D. 0.5 giây
Nhiệm vụ: Chọn đúng 1 đáp án.
Suy luận:
Hằng số thời gian RC là \tau = R * C = 500 * 100 * 10^-6 = 0.05 giây.
Vậy đáp án đúng là A.

Ví dụ 2:
Câu hỏi: Hàm cầu QD = 200 - 5P, hàm cung QS = -50 + 5P. Giá cân bằng là bao nhiêu?
Lựa chọn:
A. 20
B. 25
C. 30
D. 35
Nhiệm vụ: Chọn đúng 1 đáp án.
Suy luận:
Tại cân bằng: QD = QS <=> 200 - 5P = -50 + 5P <=> 10P = 250 <=> P = 25.
Vậy đáp án đúng là B."""

def is_reasoning_question(item):
    q_text = item["question"]
    choices = item["choices"]
    q_lower = q_text.lower()
    
    # 1. Passage-based questions are always knowledge in this dataset
    passage_prefixes = ["đoạn thông tin:", "đoạn văn:", "title:", "content:", "-- document"]
    if any(q_lower.startswith(p) for p in passage_prefixes):
        return False
        
    # 2. Check for LaTeX math symbols in question
    if "$" in q_text:
        return True

    # 3. Check for numerical/math choices
    def is_numerical_choice(choice):
        text = str(choice).strip()
        if text.startswith("$") and text.endswith("$"):
            return True
        if any(op in text for op in [r"\frac", r"\equiv", r"\pmod", r"\times", r"\sqrt", "^"]):
            return True
        clean = re.sub(r"\s*(g|ml|l|m|cm|kg|usd|đô la|%|cm/s|cm/giây|đơn vị|thaler|đáp án khác|tất cả|cả|đều|giây|ngày|tháng|năm|nguyên tắc|tiến sĩ|lần|độ)\.?$", "", text, flags=re.IGNORECASE)
        clean = clean.replace(",", ".").strip()
        if re.match(r"^[-+]?\d*\.?\d+$", clean):
            return True
        if any(c.isdigit() for c in text):
            if any(op in text for op in ["=", "+", "*", "/", "^"]):
                return True
        return False

    num_choices = [c for c in choices if is_numerical_choice(c)]
    pct_numerical = len(num_choices) / len(choices) if choices else 0
    
    # 4. Calculation/Math keywords
    math_calc_keywords = [
        "tính toán", "tính đạo hàm", "tính tích phân", "tính giới hạn", "tính nguyên hàm",
        "phương trình vi phân", "độ co giãn", "tỷ lệ lạm phát", "gdp danh nghĩa", "gdp thực tế",
        "tốc độ tăng", "tốc độ thay đổi", "điện trở tương đương", "hòa tan", "nồng độ mol",
        "nồng độ phần trăm", "trung hoà dung dịch", "lượng đặt hàng", "giá trị của biểu thức",
        "tổng chi phí", "chi phí biên", "doanh thu biên", "hàm cung", "hàm cầu", "hàm chi phí",
        "hàm sản xuất", "hàm tiêu dùng", "đường đẳng lượng", "số nhân tiền", "eoq", "định luật",
        "biến ngẫu nhiên", "xác suất", "đạo hàm bậc", "tính giá trị", "nghiệm của phương trình",
        "phương trình hoành độ", "tính số", "tốc độ đồng hồ", "lãi kép", "chi phí", "doanh thu",
        "lợi nhuận", "sản lượng", "tốc độ", "gia tốc", "chu kỳ", "tần số", "bán kính", "chiều cao",
        "thể tích", "diện tích", "số dư tài khoản", "khối lượng", "nồng độ", "điện trở", "công suất"
    ]
    has_math_keyword = any(w in q_lower for w in math_calc_keywords)
    
    # Check if the question asks for a calculation value ("bao nhiêu")
    asks_how_much = "bao nhiêu" in q_lower or "thay đổi như thế nào" in q_lower

    # Exclude keywords that indicate lookup of fact, law, history, names, administrative/legal procedures
    legal_history_keywords = [
        "luật", "quy định", "thủ tục", "thời hạn", "chính sách", "nghị định", "thông tư", "điều bộ luật",
        "cấp phép", "chứng nhận", "bảo hiểm", "năm nào", "sinh năm", "mất năm", "ai là", "quốc gia nào",
        "tác giả", "tác phẩm", "nhân vật", "ngày nào", "sự kiện", "lịch sử", "khái niệm", "định nghĩa",
        "thế kỷ", "triều đại", "ngôi chùa", "khai dựng", "biểu hiện của biến đổi khí hậu"
    ]
    has_exclude_keyword = any(w in q_lower for w in legal_history_keywords)

    # 5. Let's decide
    if pct_numerical >= 0.5:
        is_all_years = all(re.match(r"^(năm\s+)?\d{4}$", str(c).strip().lower()) for c in choices)
        if is_all_years:
            return False
        if has_exclude_keyword and not has_math_keyword and not asks_how_much:
            return False
        return True

    any_digits_in_choices = any(any(c.isdigit() for c in str(ch)) for ch in choices)
    
    if has_math_keyword and any_digits_in_choices:
        if has_exclude_keyword:
            if any(w in q_lower for w in ["tính", "tính toán", "phương trình", "hàm số", "độ co giãn", "tỷ lệ"]):
                return True
            return False
        return True

    if asks_how_much and any_digits_in_choices:
        if has_exclude_keyword:
            return False
        return True

    return False


reasoning_questions = [q for q in questions if is_reasoning_question(q)]
knowledge_questions = [q for q in questions if not is_reasoning_question(q)]

# Map chữ cái đáp án sang token ID tương ứng để tính xác suất
def get_letter_to_ids(tokenizer, choices="ABCDEFGHIJKLMNOPQRSTUVWXYZ"):
    letter_to_ids = {l: [] for l in choices}
    vocab = tokenizer.get_vocab()
    for token_str, token_id in vocab.items():
        clean_token = token_str.replace(' ', '').replace('Ġ', '').replace('\n', '').replace('<0x0A>', '').strip()
        if len(clean_token) == 1 and clean_token.upper() in choices:
            letter_to_ids[clean_token.upper()].append(token_id)
            
    for char in choices:
        for prefix in ["", " ", "\n"]:
            for c in [char, char.lower()]:
                tids = tokenizer.encode(prefix + c, add_special_tokens=False)
                if tids:
                    tid = tids[-1]
                    if tid not in letter_to_ids[char]:
                        letter_to_ids[char].append(tid)
    return letter_to_ids

# Cách 1: Tính Log-Likelihood của hậu tố (suffix) dựa trên ngữ cảnh chung (prompt) dùng KV Cache (Bản Batched Tối Ưu Tốc Độ)
def get_suffix_likelihoods_kv_cache_batched(model, processor, prompt_text, suffixes, device):
    C = len(suffixes)
    
    # 1. Tokenize prefix
    inputs_q = processor(text=prompt_text, return_tensors="pt").to(device)
    input_ids_q = inputs_q.input_ids
    attention_mask_q = inputs_q.attention_mask
    prompt_len = input_ids_q.shape[-1]
    
    # 2. Tokenize suffixes (dùng left padding cho batch)
    processor.tokenizer.padding_side = "left"
    inputs_s = processor(text=suffixes, padding=True, return_tensors="pt").to(device)
    input_ids_s = inputs_s.input_ids
    attention_mask_s = inputs_s.attention_mask
    suffix_len = input_ids_s.shape[-1]
    
    position_ids_q = torch.arange(prompt_len, dtype=torch.long, device=device).unsqueeze(0)
    
    # Tính toán position_ids cho các suffix có left padding
    cum_sum = torch.cumsum(attention_mask_s, dim=-1)
    position_ids_s = (prompt_len + cum_sum - 1) * attention_mask_s
    
    with torch.inference_mode():
        # Bước 1: Chạy forward pass phần chung để lấy KV Cache
        past_key_values = DynamicCache()
        outputs_q = model(
            input_ids=input_ids_q,
            attention_mask=attention_mask_q,
            position_ids=position_ids_q,
            past_key_values=past_key_values,
            use_cache=True
        )
        
        # Bước 2: Nhân bản KV Cache C lần trên GPU
        batched_cache = DynamicCache()
        batched_cache._seen_tokens = past_key_values.get_seq_length()
        for layer_idx in range(len(past_key_values.key_cache)):
            k = past_key_values.key_cache[layer_idx]
            v = past_key_values.value_cache[layer_idx]
            batched_cache.key_cache.append(k.repeat(C, 1, 1, 1))
            batched_cache.value_cache.append(v.repeat(C, 1, 1, 1))
            
        # Bước 3: Chạy forward pass cho các suffix song song (Batched)
        new_attention_mask = torch.cat([
            attention_mask_q.repeat(C, 1),
            attention_mask_s
        ], dim=-1)
        
        outputs_s = model(
            input_ids=input_ids_s,
            attention_mask=new_attention_mask,
            position_ids=position_ids_s,
            past_key_values=batched_cache,
            use_cache=True
        )
        suffix_logits = outputs_s.logits
        
    # Bước 4: Trích xuất log-probabilities
    prompt_logits = outputs_q.logits[:, -1:, :].repeat(C, 1, 1)
    eval_logits = torch.cat([prompt_logits, suffix_logits[:, :-1, :]], dim=1)
    log_probs = torch.log_softmax(eval_logits, dim=-1)
    
    target_log_probs = torch.gather(log_probs, dim=-1, index=input_ids_s.unsqueeze(-1)).squeeze(-1)
    target_log_probs = target_log_probs * attention_mask_s
    
    scores = []
    for i in range(C):
        num_tokens = attention_mask_s[i].sum().item()
        total_log_prob = target_log_probs[i].sum().item()
        score = total_log_prob / (num_tokens + 1e-5)
        scores.append(score)
        
    return scores

# =====================================================
# HÀM CHẠY KNOWLEDGE SỬ DỤNG LIKELIHOOD & RETRY FALLBACK
# =====================================================
def run_knowledge_generation(question_items, model, device, processor):
    if not question_items: return {}
    
    def process_batch(batch):
        preds = {}
        batch_prompts = []
        
        for item in batch:
            choices_text = "\n".join(f"{LETTERS[k]}. {choice}" for k, choice in enumerate(item["choices"]))
            user_prompt = f"Câu hỏi:\n{item['question']}\n\nLựa chọn:\n{choices_text}\n\nNhiệm vụ: Chọn đúng 1 đáp án."
            messages = [
                {"role": "system", "content": [{"type": "text", "text": SYSTEM_KNOWLEDGE}]},
                {"role": "user", "content": [{"type": "text", "text": user_prompt}]}
            ]
            prompt_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            batch_prompts.append(prompt_text + "Đáp án:")
            
        inputs = processor(text=batch_prompts, padding=True, return_tensors="pt").to(device)
        with torch.inference_mode():
            outputs = model.generate(
                **inputs, 
                max_new_tokens=15, 
                do_sample=False, 
                use_cache=True,
                pad_token_id=processor.tokenizer.eos_token_id
            )
            
        for idx, item in enumerate(batch):
            decoded = processor.tokenizer.decode(outputs[idx][inputs.input_ids.shape[1]:], skip_special_tokens=True)
            raw = decoded.strip()
            
            # Extract first uppercase letter (A-J) from the generated text
            m = re.search(r"\b([A-Z])\b", raw.upper())
            if not m:
                m = re.search(r"([A-Z])", raw.upper())
                
            pred_letter = m.group(1) if m else "A"
            preds[item["qid"]] = {"pred": pred_letter, "raw": f"Generated: '{raw}' -> Selected: {pred_letter}"}
            
        if 'inputs' in locals(): del inputs
        if 'outputs' in locals(): del outputs
        
        return preds

    # Create dynamic batches for knowledge (to avoid CUDA OOM)
    batches = []
    current_batch = []
    current_max_len = 0
    max_tokens_limit = 2500
    
    for item in question_items:
        choices_text = "\n".join(item["choices"])
        estimated_len = (len(item["question"]) + len(choices_text)) // 3
        temp_max_len = max(current_max_len, estimated_len)
        temp_batch_size = len(current_batch) + 1
        
        if (temp_batch_size * temp_max_len > max_tokens_limit or temp_batch_size > 8) and current_batch:
            batches.append(current_batch)
            current_batch = [item]
            current_max_len = estimated_len
        else:
            current_batch.append(item)
            current_max_len = temp_max_len
            
    if current_batch:
        batches.append(current_batch)
        
    results = {}
    
    def process_batch_with_fallback(batch):
        try:
            return process_batch(batch)
        except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
            if isinstance(e, RuntimeError) and "out of memory" not in str(e).lower():
                raise e
            print(f"⚠️ [OOM] Hết bộ nhớ khi sinh batch size {len(batch)} cho Knowledge. Đang tự động chia đôi...")
            torch.cuda.empty_cache()
            gc.collect()
            
            if len(batch) <= 1:
                print(f"❌ [OOM Critical] Lỗi bộ nhớ trên câu Knowledge {batch[0]['qid']}! Gán đáp án mặc định A.")
                return {batch[0]["qid"]: {"pred": "A", "raw": "OutOfMemoryError fallback default A"}}
                
            mid = len(batch) // 2
            res1 = process_batch_with_fallback(batch[:mid])
            res2 = process_batch_with_fallback(batch[mid:])
            res1.update(res2)
            return res1

    for batch in batches:
        results.update(process_batch_with_fallback(batch))
        
    return results


# =====================================================
# HÀM CHẠY REASONING SỬ DỤNG LIKELIHOOD & RETRY FALLBACK
# =====================================================
def run_reasoning_likelihood(question_items, max_new_tokens, model, device, processor):
    if not question_items: return {}
    
    letter_to_ids = get_letter_to_ids(processor.tokenizer, LETTERS)
    
    def process_batch(batch):
        preds = {}
        batch_prompts = []
        batch_prompts_dummy = []
        
        for item in batch:
            choices_text = "\n".join(f"{LETTERS[k]}. {choice}" for k, choice in enumerate(item["choices"]))
            
            # Step 1: user prompt to generate reasoning
            user_prompt = f"Câu hỏi:\n{item['question']}\n\nLựa chọn:\n{choices_text}\n\nNhiệm vụ: Chọn đúng 1 đáp án."
            
            messages = [
                {"role": "system", "content": [{"type": "text", "text": SYSTEM_REASONING}]},
                {"role": "user", "content": [{"type": "text", "text": user_prompt}]}
            ]
            prompt_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) + "Suy luận:\n"
            batch_prompts.append(prompt_text)
            
            if USE_CALIBRATION:
                dummy_prompt = f"Câu hỏi:\nN/A\n\nLựa chọn:\n{choices_text}\n\nNhiệm vụ: Chọn đúng 1 đáp án."
                messages_dummy = [
                    {"role": "system", "content": [{"type": "text", "text": SYSTEM_REASONING}]},
                    {"role": "user", "content": [{"type": "text", "text": dummy_prompt}]}
                ]
                prompt_text_dummy = processor.apply_chat_template(messages_dummy, tokenize=False, add_generation_prompt=True) + "Suy luận:\n"
                batch_prompts_dummy.append(prompt_text_dummy)
                
        # --- BƯỚC 1: SINH SUY LUẬN (BATCHED GENERATION) ---
        inputs = processor(text=batch_prompts, padding=True, return_tensors="pt").to(device)
        with torch.inference_mode():
            outputs_step1 = model.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True, pad_token_id=processor.tokenizer.eos_token_id
            )
            
        reasoning_texts = []
        for idx in range(len(batch)):
            decoded = processor.tokenizer.decode(outputs_step1[idx][inputs.input_ids.shape[1]:], skip_special_tokens=True)
            reasoning_texts.append(decoded.strip())
            
        # Giải phóng bộ nhớ Step 1
        if 'inputs' in locals(): del inputs
        if 'outputs_step1' in locals(): del outputs_step1
        
        # --- BƯỚC 2: ÉP CHỐT ĐÁP ÁN ---
        if USE_SINGLE_PASS:
            forced_texts = []
            forced_texts_dummy = []
            for idx, item in enumerate(batch):
                full_context = batch_prompts[idx] + reasoning_texts[idx] + "\n\nVậy đáp án đúng là:"
                forced_texts.append(full_context)
                
                if USE_CALIBRATION:
                    full_context_dummy = batch_prompts_dummy[idx] + "[N/A]" + "\n\nVậy đáp án đúng là:"
                    forced_texts_dummy.append(full_context_dummy)
                    
            inputs_step2 = processor(text=forced_texts, padding=True, return_tensors="pt").to(device)
            with torch.inference_mode():
                outputs_step2 = model(**inputs_step2)
                logits_step2 = outputs_step2.logits[:, -1, :]
            probs_step2 = torch.softmax(logits_step2, dim=-1)
            
            if USE_CALIBRATION:
                inputs_step2_dummy = processor(text=forced_texts_dummy, padding=True, return_tensors="pt").to(device)
                with torch.inference_mode():
                    outputs_step2_dummy = model(**inputs_step2_dummy)
                    logits_step2_dummy = outputs_step2_dummy.logits[:, -1, :]
                probs_step2_dummy = torch.softmax(logits_step2_dummy, dim=-1)
                
            for idx, item in enumerate(batch):
                choice_probs = {}
                active_letters = LETTERS[:len(item["choices"])]
                for char in active_letters:
                    tids = letter_to_ids[char]
                    score = probs_step2[idx, tids].sum().item() if tids else 0.0
                    
                    if USE_CALIBRATION:
                        score_dummy = probs_step2_dummy[idx, tids].sum().item() if tids else 0.0
                        score = score / (score_dummy + 1e-5)
                        
                    choice_probs[char] = score
                    
                total_prob = sum(choice_probs.values())
                if total_prob > 0:
                    choice_probs = {k: v / total_prob for k, v in choice_probs.items()}
                    
                best_letter = max(choice_probs, key=choice_probs.get)
                probs_str = ", ".join(f"{k}: {v:.1%}" for k, v in choice_probs.items())
                raw_combined = reasoning_texts[idx] + f"\n\n[ÉP CHỐT LIKELIHOOD]: {best_letter} ({probs_str})"
                preds[item["qid"]] = {"pred": best_letter, "raw": raw_combined}
                
            if 'inputs_step2' in locals(): del inputs_step2
            if 'outputs_step2' in locals(): del outputs_step2
            if 'inputs_step2_dummy' in locals(): del inputs_step2_dummy
            if 'outputs_step2_dummy' in locals(): del outputs_step2_dummy
            
        else:
            for idx, item in enumerate(batch):
                full_context = batch_prompts[idx] + reasoning_texts[idx] + "\n\nVậy đáp án đúng là:"
                active_letters = LETTERS[:len(item["choices"])]
                suffixes = [f" {l}" for l in active_letters]
                
                scores = get_suffix_likelihoods_kv_cache_batched(model, processor, full_context, suffixes, device)
                
                if USE_CALIBRATION:
                    full_context_dummy = batch_prompts_dummy[idx] + "[N/A]" + "\n\nVậy đáp án đúng là:"
                    scores_dummy = get_suffix_likelihoods_kv_cache_batched(model, processor, full_context_dummy, suffixes, device)
                    calibrated_scores = [s - sd for s, sd in zip(scores, scores_dummy)]
                else:
                    calibrated_scores = scores
                    
                best_idx = calibrated_scores.index(max(calibrated_scores))
                best_letter = active_letters[best_idx]
                
                exp_scores = []
                for s in calibrated_scores:
                    try:
                        exp_scores.append(math.exp(s))
                    except OverflowError:
                        exp_scores.append(float('inf'))
                sum_exp = sum(exp_scores)
                if sum_exp > 0:
                    probs_lh = [e / sum_exp for e in exp_scores]
                else:
                    probs_lh = [1.0 / len(calibrated_scores)] * len(calibrated_scores)
                    
                probs_str = ", ".join(f"{active_letters[k]}: {probs_lh[k]:.1%}" for k in range(len(active_letters)))
                raw_combined = reasoning_texts[idx] + f"\n\n[ÉP CHỐT LIKELIHOOD]: {best_letter} ({probs_str})"
                preds[item["qid"]] = {"pred": best_letter, "raw": raw_combined}
                
        return preds

    # Tạo batch động cho reasoning (Giới hạn tối đa batch size 2 do sinh chuỗi dài)
    batches = []
    current_batch = []
    current_max_len = 0
    max_tokens_limit = 2400
    
    for item in question_items:
        choices_text = "\n".join(item["choices"])
        estimated_len = (len(item["question"]) + len(choices_text)) // 3
        temp_max_len = max(current_max_len, estimated_len)
        temp_batch_size = len(current_batch) + 1
        
        if (temp_batch_size * temp_max_len > max_tokens_limit or temp_batch_size > 4) and current_batch:
            batches.append(current_batch)
            current_batch = [item]
            current_max_len = estimated_len
        else:
            current_batch.append(item)
            current_max_len = temp_max_len
            
    if current_batch:
        batches.append(current_batch)
        
    results = {}
    
    def process_batch_with_fallback(batch):
        try:
            return process_batch(batch)
        except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
            if isinstance(e, RuntimeError) and "out of memory" not in str(e).lower():
                raise e
            print(f"⚠️ [OOM] Hết bộ nhớ khi chạy Reasoning batch size {len(batch)}. Đang tự động chia đôi...")
            torch.cuda.empty_cache()
            gc.collect()
            
            if len(batch) <= 1:
                print(f"❌ [OOM Critical] Lỗi bộ nhớ nghiêm trọng trên câu Reasoning {batch[0]['qid']}! Đặt đáp án mặc định A.")
                return {batch[0]["qid"]: {"pred": "A", "raw": "OutOfMemoryError reasoning fallback default A"}}
                
            mid = len(batch) // 2
            res1 = process_batch_with_fallback(batch[:mid])
            res2 = process_batch_with_fallback(batch[mid:])
            res1.update(res2)
            return res1

    for batch in batches:
        results.update(process_batch_with_fallback(batch))
        
    return results

# =====================================================
# TRÌNH QUẢN LÝ ĐA LUỒNG CHO PHƯƠNG PHÁP MỚI
# =====================================================
def run_batch_likelihood(question_items, mode, max_new_tokens):
    if not question_items: return {}

    num_workers = len(models)
    chunk_size = (len(question_items) + num_workers - 1) // num_workers
    chunks = [question_items[i:i + chunk_size] for i in range(0, len(question_items), chunk_size)]
    
    results = {}
    
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        futures = []
        for i, chunk in enumerate(chunks):
            if chunk:
                target_model = models[i % num_workers]
                target_device = devices[i % num_workers]
                target_processor = processors[i % num_workers]
                
                if mode == "KNOWLEDGE":
                    futures.append(executor.submit(run_knowledge_generation, chunk, target_model, target_device, target_processor))
                else:
                    futures.append(executor.submit(run_reasoning_likelihood, chunk, max_new_tokens, target_model, target_device, target_processor))
                
        for future in futures:
            results.update(future.result())
            
    return results

# =====================================================
# THỰC THI PHƯƠNG PHÁP MỚI & HIỂN THỊ
# =====================================================
results_lh = {}

print(f"\n[1/2] Đang xử lý {len(knowledge_questions)} câu hỏi thường (Likelihood)...")
if knowledge_questions:
    results_lh.update(run_batch_likelihood(knowledge_questions, mode="KNOWLEDGE", max_new_tokens=1))

print(f"\n[2/2] Đang xử lý {len(reasoning_questions)} câu hỏi 'là bao nhiêu' (Likelihood + Reasoning)...")
if reasoning_questions:
    results_lh.update(run_batch_likelihood(reasoning_questions, mode="REASONING", max_new_tokens=300))

# =====================================================
# HIỂN THỊ KẾT QUẢ ĐÁNH GIÁ
# =====================================================
print("\n" + "=" * 100)
print("MODEL OUTPUT (LIKELIHOOD METHOD)")
print("=" * 100)

for item in questions:
    qid = item["qid"]
    mode = "REASONING" if is_reasoning_question(item) else "KNOWLEDGE"
    res = results_lh.get(qid, {"pred": "?", "raw": "NO OUTPUT"})
    
    print(f"QID   : {qid}")
    print(f"MODE  : {mode}")
    print(f"RAW   :\n{res['raw']}")
    print(f"PRED  : {res['pred']}")
    print("-" * 100)

correct_lh = 0
total_evaluated_lh = 0
print("\n" + "=" * 100 + "\nEVALUATION (LIKELIHOOD METHOD)\n" + "=" * 100)
for item in questions:
    qid = item["qid"]
    if qid in GROUND_TRUTH:
        gt = GROUND_TRUTH[qid]
        pred = results_lh.get(qid, {}).get("pred", "?")
        ok = pred == gt
        if ok:
            correct_lh += 1
        total_evaluated_lh += 1
        print(f"{qid:12s} | Pred={pred:2s} | GT={gt:2s} | {'✓' if ok else '✗'}")

print("=" * 100)
if total_evaluated_lh > 0:
    print(f"Correct : {correct_lh}/{total_evaluated_lh}\nAccuracy: {correct_lh / total_evaluated_lh:.2%}")
else:
    print("No questions with ground truth were evaluated.")
print("=" * 100)


[1/2] Đang xử lý 170 câu hỏi thường (Likelihood)...

[2/2] Đang xử lý 93 câu hỏi 'là bao nhiêu' (Likelihood + Reasoning)...


IndexError: string index out of range